In [ ]:
# ==============================================================================
# Cell 1: Setup - Clone repo, download GSC V2, install deps
# ==============================================================================
import os, sys

# Clone repo (or pull latest)
if os.path.exists('/content/NC-KWS-FineTuning'):
    !cd /content/NC-KWS-FineTuning && git pull
else:
    !cd /content && git clone https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git

%cd /content/NC-KWS-FineTuning
!pip install -q torch torchaudio numpy matplotlib pandas

sys.path.insert(0, '/content/NC-KWS-FineTuning')

# Verify
assert os.path.exists('nanomamba.py'), 'nanomamba.py missing!'
assert os.path.exists('checkpoints/best.pt'), 'checkpoint missing!'
print('Repo OK')

# Download GSC V2
import tarfile, urllib.request
GSC_URL = 'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
GSC_TAR = 'data/speech_commands_v0.02.tar.gz'
GSC_DIR = 'data/speech_commands_v0.02'

os.makedirs('data', exist_ok=True)
if not os.path.exists(GSC_DIR):
    if not os.path.exists(GSC_TAR):
        print('Downloading GSC V2 (~2.3GB)...')
        urllib.request.urlretrieve(GSC_URL, GSC_TAR)
    print('Extracting...')
    os.makedirs(GSC_DIR, exist_ok=True)
    with tarfile.open(GSC_TAR, 'r:gz') as tar:
        tar.extractall(GSC_DIR)
    print('Done!')
else:
    print(f'GSC V2 ready at {GSC_DIR}')

kw_dirs = sorted([d for d in os.listdir(GSC_DIR)
                  if os.path.isdir(os.path.join(GSC_DIR, d)) and not d.startswith('_')])
print(f'{len(kw_dirs)} keywords found')

In [ ]:
# ==============================================================================
# Cell 2: Full Experiment - All 3 methods, train + eval in one cell
# ==============================================================================
import os, sys, time, tempfile, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
from collections import defaultdict

os.chdir('/content/NC-KWS-FineTuning')
sys.path.insert(0, '/content/NC-KWS-FineTuning')

np.random.seed(42)
torch.manual_seed(42)

# ── Config ──
SR = 16000
GSC_DIR = 'data/speech_commands_v0.02'
BASE_LABELS = ['yes','no','up','down','left','right','on','off','stop','go','silence','unknown']
NEW_KEYWORDS = ['marvin', 'sheila', 'bed']
N_TRAIN = 20     # new keyword samples for training
N_BASE = 10      # base class samples for training (anti-forgetting!)
N_TEST = 100     # test samples per class

# ── Helper: load wav ──
def load_wav(keyword, n, offset=0):
    kw_dir = os.path.join(GSC_DIR, keyword)
    if not os.path.exists(kw_dir): return []
    files = sorted(f for f in os.listdir(kw_dir) if f.endswith('.wav'))
    samples = []
    for f in files[offset:offset+n]:
        try:
            w, sr = torchaudio.load(os.path.join(kw_dir, f))
            a = w[0].numpy()
            if len(a) > SR: a = a[:SR]
            elif len(a) < SR: a = np.pad(a, (0, SR-len(a)))
            samples.append(a.astype(np.float32))
        except: continue
    return samples

# ── Load data ──
print('Loading training data (new keywords)...')
train_new = {}
for kw in NEW_KEYWORDS:
    train_new[kw] = load_wav(kw, N_TRAIN)
    print(f'  {kw}: {len(train_new[kw])}')

print('Loading training data (base classes for anti-forgetting)...')
train_base = {}
base_words = ['yes','no','up','down','left','right','on','off','stop','go']
for kw in base_words:
    train_base[kw] = load_wav(kw, N_BASE, offset=0)
    print(f'  {kw}: {len(train_base[kw])}')

print('Loading test data...')
test_data = {}
for kw in NEW_KEYWORDS:
    test_data[kw] = load_wav(kw, N_TEST, offset=N_TRAIN+200)
for kw in base_words:
    test_data[kw] = load_wav(kw, N_TEST, offset=500)
test_data['silence'] = [np.random.randn(SR).astype(np.float32)*0.002 for _ in range(N_TEST)]
unk_words = ['bird','cat','dog','happy','house','tree','wow']
unk = []
for w in unk_words:
    unk.extend(load_wav(w, N_TEST//len(unk_words), offset=300))
test_data['unknown'] = unk[:N_TEST]
for kw in test_data:
    print(f'  {kw}: {len(test_data[kw])} test')

# ── Model + Evaluate ──
from nanomamba import create_nc_tcn_20k

def evaluate(model, labels, test_data):
    model.eval()
    results = {}
    tc, tn = 0, 0
    with torch.no_grad():
        for kw, samples in test_data.items():
            ti = labels.index(kw) if kw in labels else labels.index('unknown')
            c = sum(1 for a in samples
                    if model(torch.from_numpy(a).float().unsqueeze(0)).argmax(-1).item() == ti)
            results[kw] = c / max(len(samples),1) * 100
            tc += c; tn += len(samples)
    results['_overall'] = tc / max(tn,1) * 100
    return results

# ── Baseline ──
print('\n' + '='*60)
print('  BASELINE (no fine-tuning)')
print('='*60)
base_model = create_nc_tcn_20k()
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
base_model.load_state_dict(ckpt['model_state_dict'], strict=False)
base_eval = evaluate(base_model, BASE_LABELS, test_data)
for kw in NEW_KEYWORDS + base_words + ['silence','unknown']:
    tag = ' *NEW*' if kw in NEW_KEYWORDS else ''
    print(f'  {kw:<12} {base_eval.get(kw,0):>6.1f}%{tag}')
print(f'  {"OVERALL":<12} {base_eval["_overall"]:>6.1f}%')

# =====================================================================
# LoRA class (shared by Standard LoRA and NC-OPAL)
# =====================================================================
class LoRALinear(nn.Module):
    def __init__(self, original, rank=4, alpha=8):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

# =====================================================================
# Prepare training data (KEY FIX: include real base class audio!)
# =====================================================================
def prepare_train_data(train_new, train_base, base_labels, custom_labels):
    """Build training set with REAL base class audio + new keyword audio."""
    X, Y = [], []
    n_base = len(base_labels)

    # New keyword samples (3x augmentation)
    for i, kw in enumerate(custom_labels):
        li = n_base + i
        for a in train_new.get(kw, []):
            X.append(a); Y.append(li)
            X.append(np.roll(a, np.random.randint(-1600,1600))); Y.append(li)
            X.append(a + np.random.randn(len(a)).astype(np.float32)*0.005); Y.append(li)

    # REAL base class audio (KEY: prevents catastrophic forgetting!)
    base_word_list = ['yes','no','up','down','left','right','on','off','stop','go']
    for kw in base_word_list:
        li = base_labels.index(kw)
        for a in train_base.get(kw, []):
            X.append(a); Y.append(li)
            X.append(a + np.random.randn(len(a)).astype(np.float32)*0.003); Y.append(li)

    # Synthetic silence/unknown
    n_custom = sum(len(train_new[k]) for k in custom_labels) * 3
    n_neg = n_custom
    si, ui = base_labels.index('silence'), base_labels.index('unknown')
    for _ in range(n_neg//2):
        X.append(np.random.randn(SR).astype(np.float32)*0.001); Y.append(si)
    for _ in range(n_neg - n_neg//2):
        X.append(np.random.randn(SR).astype(np.float32)*0.01); Y.append(ui)

    print(f'  Training data: {len(X)} samples '
          f'(new={n_custom}, base={sum(2*len(train_base.get(k,[])) for k in base_word_list)}, neg={n_neg})')
    return X, Y

# =====================================================================
# METHOD 1: Standard LoRA
# =====================================================================
def run_standard_lora(train_new, train_base, epochs=20, lr=3e-3):
    print('\n' + '='*60)
    print('  METHOD 1: Standard LoRA (rank=4)')
    print('='*60)

    model = create_nc_tcn_20k()
    ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)

    custom = list(train_new.keys())
    n_base = len(BASE_LABELS)
    n_total = n_base + len(custom)
    all_labels = BASE_LABELS + custom

    # Expand head
    old_head = model.classifier
    d = old_head.in_features
    new_head = nn.Linear(d, n_total)
    with torch.no_grad():
        new_head.weight[:n_base] = old_head.weight
        new_head.bias[:n_base] = old_head.bias
        nn.init.xavier_uniform_(new_head.weight[n_base:])
        new_head.bias[n_base:].zero_()
    model.classifier = new_head

    # Apply LoRA
    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                lo = LoRALinear(getattr(block, attr), rank=4)
                setattr(block, attr, lo); loras.append(lo)

    # Freeze all, unfreeze LoRA + head
    for p in model.parameters(): p.requires_grad = False
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    for p in new_head.parameters(): p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable: {trainable:,}')

    X, Y = prepare_train_data(train_new, train_base, BASE_LABELS, custom)

    # Train
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    t0 = time.time()
    for ep in range(epochs):
        perm = np.random.permutation(len(X))
        el, nb = 0, 0
        for i in range(0, len(X), 8):
            bi = perm[i:i+8]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi])
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); el += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 5 == 0 or ep == 0:
            print(f'  Epoch {ep+1}/{epochs} loss={el/nb:.4f}')

    print(f'  Time: {time.time()-t0:.1f}s')
    return model, all_labels

# =====================================================================
# METHOD 2: NC-OPAL (Prototype-Imprinted Init + KD)
# =====================================================================
def run_ncopal(train_new, train_base, epochs=20, lr=3e-3, lambda_kd=1.0, kd_temp=4.0):
    print('\n' + '='*60)
    print('  METHOD 3: NC-OPAL (Prototype Init + KD)')
    print('='*60)

    model = create_nc_tcn_20k()
    ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
    state = ckpt['model_state_dict']
    model.load_state_dict(state, strict=False)

    custom = list(train_new.keys())
    n_base = len(BASE_LABELS)
    n_total = n_base + len(custom)
    all_labels = BASE_LABELS + custom
    d = model.classifier.in_features

    # [NOVELTY 1] Prototype-Imprinted Head
    print('  Prototype-imprinted head init...')
    model.eval()
    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(audio):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(audio).float().unsqueeze(0))
        h.remove()
        return emb_hook['e'].squeeze(0)

    old_head = model.classifier
    new_head = nn.Linear(d, n_total)
    with torch.no_grad():
        new_head.weight[:n_base] = old_head.weight
        new_head.bias[:n_base] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, kw in enumerate(custom):
            embs = [get_emb(a) for a in train_new[kw]]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_base+i] = proto * scale
                new_head.bias[n_base+i] = old_head.bias.mean().item()
    model.classifier = new_head

    # Apply LoRA
    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                lo = LoRALinear(getattr(block, attr), rank=4)
                setattr(block, attr, lo); loras.append(lo)

    for p in model.parameters(): p.requires_grad = False
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    for p in new_head.parameters(): p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable: {trainable:,}')

    # [NOVELTY 2] Frozen teacher for KD
    teacher = create_nc_tcn_20k()
    teacher.load_state_dict(state, strict=False)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    print(f'  KD teacher loaded (lambda={lambda_kd})')

    X, Y = prepare_train_data(train_new, train_base, BASE_LABELS, custom)

    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    t0 = time.time()
    for ep in range(epochs):
        perm = np.random.permutation(len(X))
        ecl, ekd, nb = 0, 0, 0
        for i in range(0, len(X), 8):
            bi = perm[i:i+8]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi])
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
            logits = model(bx)
            cls_loss = F.cross_entropy(logits, by)

            # KD on base-class samples only
            kd_loss = torch.tensor(0.0)
            base_mask = by < n_base
            if base_mask.any() and lambda_kd > 0:
                with torch.no_grad(): t_logits = teacher(bx[base_mask])
                s_base = logits[base_mask][:, :n_base] / kd_temp
                t_base = t_logits / kd_temp
                kd_loss = F.kl_div(F.log_softmax(s_base, -1),
                                   F.softmax(t_base, -1),
                                   reduction='batchmean') * (kd_temp**2)

            total = cls_loss + lambda_kd * kd_loss
            opt.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step()
            ecl += cls_loss.item(); ekd += kd_loss.item(); nb += 1
        sched.step()
        if (ep+1) % 5 == 0 or ep == 0:
            print(f'  Epoch {ep+1}/{epochs} cls={ecl/nb:.4f} kd={ekd/nb:.4f}')

    print(f'  Time: {time.time()-t0:.1f}s')
    return model, all_labels

# =====================================================================
# METHOD 3: NC-ALoRA-PR (Adaptive LoRA + Prototype Regularization)
# =====================================================================
class AdaptiveLoRALinear(nn.Module):
    def __init__(self, original, max_rank=8, min_rank=1, alpha=16.0):
        super().__init__()
        self.original = original
        self.max_rank = max_rank; self.min_rank = min_rank; self.alpha = alpha
        self.lora_A = nn.Parameter(torch.randn(original.in_features, max_rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(max_rank, original.out_features))
        self.rank_importance = nn.Parameter(torch.ones(max_rank))
        self._effective_rank = max_rank; self._mask = None
    @property
    def scaling(self): return self.alpha / self._effective_rank
    def forward(self, x):
        gate = torch.sigmoid(self.rank_importance)
        if self._mask is not None: gate = gate * self._mask
        return self.original(x) + (x @ (self.lora_A * gate.unsqueeze(0)) @ self.lora_B) * self.scaling
    def prune_rank(self, threshold=0.1):
        with torch.no_grad():
            imp = torch.sigmoid(self.rank_importance)
            mask = (imp > threshold).float()
            if mask.sum() < self.min_rank:
                _, topk = imp.topk(self.min_rank)
                mask = torch.zeros_like(mask); mask[topk] = 1.0
            self._mask = mask
            self._effective_rank = max(int(mask.sum().item()), self.min_rank)
        return self._effective_rank

def run_ncalora_pr(train_new, train_base, epochs=25, lr=3e-3, prune_ep=10):
    print('\n' + '='*60)
    print('  METHOD 2: NC-ALoRA-PR (Adaptive rank + Prototype Reg.)')
    print('='*60)

    model = create_nc_tcn_20k()
    ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)

    custom = list(train_new.keys())
    n_base = len(BASE_LABELS)
    n_total = n_base + len(custom)
    all_labels = BASE_LABELS + custom
    d = model.classifier.in_features

    old_head = model.classifier
    new_head = nn.Linear(d, n_total)
    with torch.no_grad():
        new_head.weight[:n_base] = old_head.weight
        new_head.bias[:n_base] = old_head.bias
        nn.init.xavier_uniform_(new_head.weight[n_base:])
        new_head.bias[n_base:].zero_()
    model.classifier = new_head

    # Adaptive LoRA
    aloras, anames = [], []
    for bi, block in enumerate(model.blocks):
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                a = AdaptiveLoRALinear(getattr(block, attr), max_rank=8, alpha=16)
                setattr(block, attr, a); aloras.append(a); anames.append(f'b{bi}.{attr}')

    for p in model.parameters(): p.requires_grad = False
    for m in aloras:
        for p in m.parameters(): p.requires_grad = True
    for p in new_head.parameters(): p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable: {trainable:,}, ALoRA modules: {len(aloras)}')

    # Build prototypes (from REAL audio!)
    print('  Building prototypes from real audio...')
    model.eval()
    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0]
    def fwd_emb(audio_batch):
        h = model.classifier.register_forward_hook(hook_fn)
        logits = model(audio_batch); h.remove()
        return logits, emb_hook.get('e', torch.zeros(audio_batch.shape[0], d))

    protos = torch.zeros(n_total, d)
    counts = torch.zeros(n_total)
    # Base class prototypes from real audio
    with torch.no_grad():
        for kw in base_words:
            li = BASE_LABELS.index(kw)
            samples = train_base.get(kw, [])
            if samples:
                bx = torch.stack([torch.from_numpy(a).float() for a in samples])
                _, emb = fwd_emb(bx)
                protos[li] = emb.mean(0); counts[li] = len(samples)
        # New class prototypes
        for i, kw in enumerate(custom):
            li = n_base + i
            bx = torch.stack([torch.from_numpy(a).float() for a in train_new[kw]])
            _, emb = fwd_emb(bx)
            protos[li] = emb.mean(0); counts[li] = len(train_new[kw])
    # Normalize
    norms = protos.norm(dim=1, keepdim=True).clamp(min=1e-8)
    protos = protos / norms
    print(f'  Prototypes built for {int((counts>0).sum())} classes')

    X, Y = prepare_train_data(train_new, train_base, BASE_LABELS, custom)

    # Grad SNR monitor
    grad_hist = defaultdict(list)

    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    warmup = min(5, epochs//4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs,
                                                 pct_start=warmup/epochs, anneal_strategy='cos')

    t0 = time.time()
    for ep in range(epochs):
        perm = np.random.permutation(len(X))
        el, nb = 0, 0
        for i in range(0, len(X), 8):
            bi = perm[i:i+8]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi])
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
            logits, emb = fwd_emb(bx)

            cls = F.cross_entropy(logits, by)

            # Prototype distillation (base classes)
            proto_loss = torch.tensor(0.0)
            n_p = 0
            for c in range(n_base):
                mask = by == c
                if mask.any() and counts[c] > 0:
                    cur = F.normalize(emb[mask], dim=1)
                    sim = (cur * protos[c].unsqueeze(0)).sum(1)
                    proto_loss = proto_loss + (1-sim).mean(); n_p += 1
            if n_p > 0: proto_loss = proto_loss / n_p

            # Contrastive (new vs base)
            contr_loss = torch.tensor(0.0)
            n_c = 0
            for c in range(n_base, n_total):
                mask = by == c
                if not mask.any(): continue
                cur = F.normalize(emb[mask], dim=1)
                own = protos[c].unsqueeze(0)
                pos = 1 - (cur * own).sum(1)
                neg_sim = cur @ protos[:n_base].t()
                neg = 1 - neg_sim.max(1).values
                contr_loss = contr_loss + F.relu(pos - neg + 0.5).mean(); n_c += 1
            if n_c > 0: contr_loss = contr_loss / n_c

            pw = 0.3 * min(1.0, ep / max(warmup, 1))
            total = cls + pw * proto_loss + 0.2 * contr_loss
            opt.zero_grad(); total.backward()

            for n, am in zip(anames, aloras):
                if am.lora_A.grad is not None:
                    grad_hist[n].append(am.lora_A.grad.detach().clone())
                    if len(grad_hist[n]) > 5: grad_hist[n].pop(0)

            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); el += total.item(); nb += 1

        sched.step()

        # Rank pruning
        if ep == prune_ep:
            for n, am in zip(anames, aloras):
                grads = grad_hist.get(n, [])
                snr = 1.0
                if len(grads) >= 2:
                    st = torch.stack(grads)
                    var = st.var(0).mean(); sig = (st.mean(0)**2).mean()
                    snr = (sig/(var+1e-10)).item() if var > 1e-10 else 10.0
                th = 0.3 if snr < 0.5 else 0.15 if snr < 1.0 else 0.05
                er = am.prune_rank(th)
                print(f'    {n}: SNR={snr:.2f} -> rank={er}')

        if (ep+1) % 5 == 0 or ep == 0:
            print(f'  Epoch {ep+1}/{epochs} loss={el/nb:.4f}')

    print(f'  Time: {time.time()-t0:.1f}s')
    return model, all_labels

# =====================================================================
# RUN ALL EXPERIMENTS
# =====================================================================
results = {}

# Method 1
m1, l1 = run_standard_lora(train_new, train_base)
eval1 = evaluate(m1, l1, test_data)
results['Standard LoRA'] = eval1
print('\nStandard LoRA test:')
for kw in NEW_KEYWORDS + base_words + ['silence','unknown']:
    tag = ' *NEW*' if kw in NEW_KEYWORDS else ''
    print(f'  {kw:<12} {eval1.get(kw,0):>6.1f}%{tag}')
print(f'  {"OVERALL":<12} {eval1["_overall"]:>6.1f}%')

# Method 2
m2, l2 = run_ncalora_pr(train_new, train_base)
eval2 = evaluate(m2, l2, test_data)
results['NC-ALoRA-PR'] = eval2
print('\nNC-ALoRA-PR test:')
for kw in NEW_KEYWORDS + base_words + ['silence','unknown']:
    tag = ' *NEW*' if kw in NEW_KEYWORDS else ''
    print(f'  {kw:<12} {eval2.get(kw,0):>6.1f}%{tag}')
print(f'  {"OVERALL":<12} {eval2["_overall"]:>6.1f}%')

# Method 3
m3, l3 = run_ncopal(train_new, train_base)
eval3 = evaluate(m3, l3, test_data)
results['NC-OPAL'] = eval3
print('\nNC-OPAL test:')
for kw in NEW_KEYWORDS + base_words + ['silence','unknown']:
    tag = ' *NEW*' if kw in NEW_KEYWORDS else ''
    print(f'  {kw:<12} {eval3.get(kw,0):>6.1f}%{tag}')
print(f'  {"OVERALL":<12} {eval3["_overall"]:>6.1f}%')

# =====================================================================
# COMPARISON TABLE
# =====================================================================
print('\n' + '='*70)
print(f'{"Method":<18} {"Overall":>8} {"NewKW":>8} {"BaseKW":>8} {"Forgetting":>10}')
print('-'*55)

base_kws = base_words + ['silence','unknown']
base_baseline = np.mean([base_eval.get(k,0) for k in base_kws])

for method, ev in [('Base (no FT)', base_eval)] + list(results.items()):
    overall = ev['_overall']
    new_acc = np.mean([ev.get(k,0) for k in NEW_KEYWORDS])
    base_acc = np.mean([ev.get(k,0) for k in base_kws])
    forget = base_baseline - base_acc
    print(f'{method:<18} {overall:>7.1f}% {new_acc:>7.1f}% {base_acc:>7.1f}% {forget:>+9.1f}%')

In [ ]:
# ==============================================================================
# Cell 3: Visualization
# ==============================================================================
import matplotlib.pyplot as plt
import pandas as pd

all_evals = {'Base': base_eval, 'LoRA': eval1, 'ALoRA-PR': eval2, 'NC-OPAL': eval3}

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = list(all_evals.keys())
colors = ['#888888', '#4285F4', '#EA4335', '#34A853']

# New keywords
ax = axes[0]
for i, (m, ev) in enumerate(all_evals.items()):
    accs = [ev.get(k, 0) for k in NEW_KEYWORDS]
    x = np.arange(len(NEW_KEYWORDS)) + i*0.2
    ax.bar(x, accs, 0.18, label=m, color=colors[i])
ax.set_xticks(np.arange(len(NEW_KEYWORDS)) + 0.3)
ax.set_xticklabels(NEW_KEYWORDS)
ax.set_ylabel('Accuracy (%)')
ax.set_title('New Keyword Accuracy')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# Base keywords
ax = axes[1]
for i, (m, ev) in enumerate(all_evals.items()):
    accs = [ev.get(k, 0) for k in base_words]
    ax.bar(np.arange(len(base_words)) + i*0.2, accs, 0.18, label=m, color=colors[i])
ax.set_xticks(np.arange(len(base_words)) + 0.3)
ax.set_xticklabels(base_words, rotation=45, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Base Class Retention')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# Summary
ax = axes[2]
summary_metrics = ['Overall', 'New KW', 'Base KW']
for i, (m, ev) in enumerate(all_evals.items()):
    vals = [ev['_overall'],
            np.mean([ev.get(k,0) for k in NEW_KEYWORDS]),
            np.mean([ev.get(k,0) for k in base_words + ['silence','unknown']])]
    ax.bar(np.arange(3) + i*0.2, vals, 0.18, label=m, color=colors[i])
ax.set_xticks(np.arange(3) + 0.3)
ax.set_xticklabels(summary_metrics)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Summary')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

plt.suptitle('NC-TCN-20K KWS Fine-Tuning: 3 Methods Comparison (20-shot)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print detailed table
print('\nDetailed per-keyword accuracy:')
rows = []
all_kws = NEW_KEYWORDS + base_words + ['silence', 'unknown']
for kw in all_kws:
    row = {'Keyword': kw + (' *' if kw in NEW_KEYWORDS else '')}
    for m, ev in all_evals.items():
        row[m] = f'{ev.get(kw,0):.1f}%'
    rows.append(row)
df = pd.DataFrame(rows)
print(df.to_string(index=False))